# Module 2 Chapter 6：随机平滑（Randomized Smoothing）

本 notebook 介绍 **Randomized Smoothing** 防御机制，由 Cohen et al. 在《Certified Adversarial Robustness via Randomized Smoothing》中提出。核心思想是：
- 在推理时对输入多次加入高斯噪声，统计模型预测结果；
- 出现次数最多的类别称为“平滑预测”（smoothed prediction）；
- 若最高类别置信度足够高，则可给出针对 $L_2$ 扰动的**可证明鲁棒半径**（certified radius）。

该方法仅提供 $L_2$ 范数下的认证鲁棒性，噪声标准差 $\sigma$ 越大，认证半径通常越大，但干净准确率可能下降。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
import numpy as np
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Standard normal CDF and inverse CDF (no scipy allowed)
# 标准正态分布的 CDF 与分位数函数（使用 Newton 迭代）
def norm_cdf(x):
    return 0.5 * math.erfc(-x / math.sqrt(2))


def norm_ppf(p):
    if p <= 0:
        return -float("inf")
    if p >= 1:
        return float("inf")
    # Initial guess based on tail probability
    if p < 0.5:
        x = -math.sqrt(-2 * math.log(p))
    else:
        x = math.sqrt(-2 * math.log(1 - p))
    for _ in range(50):
        f = norm_cdf(x) - p
        df = math.exp(-x * x / 2) / math.sqrt(2 * math.pi)
        if abs(df) < 1e-12:
            break
        x = x - f / df
    return x


# SimpleCNN for CIFAR-10 / 用于 CIFAR-10 的简单 CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


# Load CIFAR-10 / 加载数据
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.CIFAR10(root="./data", train=True, download=False, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")


In [ ]:
# Train base classifier / 训练基分类器
def train_classifier(model, loader, epochs, lr=0.001):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        total_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"[Base] Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(loader):.4f}")

base_model = SimpleCNN().to(device)
print("Training base classifier (5 epochs for quick demo)...")
train_classifier(base_model, train_loader, epochs=5)

# Clean accuracy / 干净准确率
base_model.eval()
correct = 0
total = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (base_model(x).argmax(1) == y).sum().item()
        total += y.size(0)
print(f"Clean test accuracy: {correct / total:.4f}")


In [ ]:
# Randomized smoothing prediction / 随机平滑预测：多数投票
def predict_smooth(model, x, num_samples, sigma):
    model.eval()
    counts = [0] * 10
    with torch.no_grad():
        for _ in range(num_samples):
            noise = torch.randn_like(x) * sigma
            x_noisy = torch.clamp(x + noise, 0.0, 1.0)  # 确保噪声图像在有效范围内
            pred = model(x + noise).argmax(1).item()
            counts[pred] += 1
    return max(range(10), key=lambda i: counts[i])


# Certified radius using normal approximation (Cohen et al. uses Clopper-Pearson; we approximate)
# 认证半径：使用正态近似计算置信下界；原 Cohen 论文使用 Clopper-Pearson（需要 scipy）
def certified_radius(model, x, num_samples, sigma, alpha=0.001):
    model.eval()
    counts = [0] * 10
    with torch.no_grad():
        for _ in range(num_samples):
            noise = torch.randn_like(x) * sigma
            x_noisy = torch.clamp(x + noise, 0.0, 1.0)  # 确保噪声图像在有效范围内
            pred = model(x + noise).argmax(1).item()
            counts[pred] += 1
    top_class = max(range(10), key=lambda i: counts[i])
    top_count = counts[top_class]
    pA = top_count / num_samples
    # Normal-approximation lower confidence bound
    z = norm_ppf(1 - alpha)
    p_lower = pA - z * math.sqrt(pA * (1 - pA) / num_samples)
    p_lower = max(0.5, min(p_lower, 1.0 - 1e-12))
    if p_lower <= 0.5:
        return 0.0, top_class
    radius = sigma * norm_ppf(p_lower)
    return radius, top_class


In [ ]:
# Demonstrate certified radius for a few images at different sigma values
# 展示不同 sigma 下若干测试样本的认证半径
num_samples = 100
sigmas = [0.0, 0.12, 0.25, 0.5, 1.0]
classes = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

for i in range(5):
    x, y = test_dataset[i]
    x = x.unsqueeze(0).to(device)
    print(f"\nImage {i}: true label = {classes[y]}")
    for sigma in sigmas:
        if sigma == 0.0:
            # No smoothing: baseline prediction
            with torch.no_grad():
                pred = base_model(x).argmax(1).item()
            radius = 0.0
        else:
            radius, pred = certified_radius(base_model, x, num_samples, sigma)
        print(f"  sigma={sigma:4.2f} -> pred={classes[pred]:10s}, certified_radius={radius:.4f}")


## 解读

σ=0 时认证半径为 0，意味着没有任何可证明的防护。σ=0.12 时半径增加到约 0.9，即任何 L2 扰动不超过 0.9 的攻击都保证被防御。但 σ=0.25 时预测标签可能从 cat 变成 automobile，说明噪声过大会损害正常分类准确率。

In [ ]:
# Certified accuracy vs L2 radius curve / 认证准确率-半径曲线
def certified_accuracy_curve(model, loader, sigma, num_samples=100, max_samples=500):
    radii = np.linspace(0, 1.0, 50)
    results = []
    n = 0
    model.eval()
    for x, y in loader:
        x = x.to(device)
        for j in range(x.size(0)):
            xj = x[j:j + 1]
            radius, pred = certified_radius(model, xj, num_samples, sigma)
            results.append((radius, pred, y[j].item()))
            n += 1
            if n >= max_samples:
                break
        if n >= max_samples:
            break
    accs = []
    for r in radii:
        correct = sum(1 for radius, pred, true in results if radius >= r and pred == true)
        accs.append(correct / len(results))
    return radii, accs


plt.figure(figsize=(10, 6))
sigmas_plot = [0.12, 0.25, 0.5, 1.0]
for sigma in sigmas_plot:
    radii, accs = certified_accuracy_curve(base_model, test_loader, sigma, num_samples=100, max_samples=500)
    plt.plot(radii, accs, label=f"sigma={sigma:.2f}")

plt.xlabel("Certified L2 Radius")
plt.ylabel("Certified Accuracy")
plt.title("Certified Accuracy vs L2 Radius (Randomized Smoothing)")
plt.legend()
plt.grid(True)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 解读

这是随机平滑的核心图表：横轴是攻击者可能施加的 L2 扰动半径，纵轴是保证的准确率。曲线越高、越宽，代表防御越强。与对抗训练（Notebook 27）不同，随机平滑提供的不是“大概率能防住”的经验鲁棒性，而是“在这个半径内一定能防住”的数学证明。

#### 不同 σ 下的认证鲁棒半径

下面展示若干测试图像在不同高斯噪声水平 σ 下的认证 L2 半径。通常 σ 越大，认证半径越大，但干净准确率可能下降。

In [ ]:
# 收集 5 张测试图像在不同 sigma 下的认证 L2 半径
num_samples = 100
sigmas_plot = [0.12, 0.25, 0.5, 1.0]
num_images = 5
radii_matrix = []

for i in range(num_images):
    x, y = test_dataset[i]
    x = x.unsqueeze(0).to(device)
    row = []
    for sigma in sigmas_plot:
        radius, pred = certified_radius(base_model, x, num_samples, sigma)
        row.append(radius)
    radii_matrix.append(row)

radii_matrix = np.array(radii_matrix)

x = np.arange(len(sigmas_plot))
width = 0.15
plt.figure(figsize=(10, 6))
for i in range(num_images):
    plt.bar(x + i * width, radii_matrix[i], width, label=f"图像 {i}")

plt.xlabel("噪声水平 σ")
plt.ylabel("认证 L2 半径")
plt.title("不同噪声水平 (σ) 下的认证鲁棒半径")
plt.xticks(x + width * (num_images - 1) / 2, [str(s) for s in sigmas_plot])
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 解读

σ 是随机平滑中准确率与鲁棒性的旋钮：σ 小 → 干净准确率高，但认证半径小；σ 大 → 认证半径大，但干净准确率下降。实践中，σ=0.25-0.5 通常是兼顾两者的选择。

## 小结与讨论

- **随机平滑**通过对输入添加高斯噪声并统计多数预测，获得针对 $L_2$ 扰动的可证明鲁棒半径。
- **sigma 越大**，认证半径越大，但干净准确率通常会下降；这是另一种形式的**鲁棒性-准确率权衡**。
- **局限性**：认证保证仅适用于 $L_2$ 攻击；对于 $L_\infty$ 攻击没有直接保证；大量采样会增加推理成本。
- **下一步**：可尝试使用更鲁棒的基础模型、调整采样数量，或结合对抗训练进一步提升认证准确率。

随机平滑是唯一能够提供可证明保证的 L2 防御方法。但它的保证仅限于 L2 攻击——对 L∞ 攻击（如 FGSM）没有认证保证。它与 Notebook 27（对抗训练，经验防御但覆盖所有攻击类型）和 Notebook 28（检测，互补但可被自适应攻击绕过）共同构成了三种不同思路的防御体系。
